# Phase 2 - Model Implementation and Training

This notebook completes model implementation, model training, hyperparameter tuning, model saving, and prediction export for hotspot forecasting.

## Model Implementation

Four scikit-learn classification models are implemented:

- Logistic Regression
- Random Forest
- Gradient Boosting
- HistGradientBoosting

These models provide a balanced comparison between a simple linear baseline and stronger non-linear ensemble methods. XGBoost was not used here because the local environment did not contain the package, while the four selected models can run directly in a fully reproducible way.

The prediction target is `target_hotspot_next_block`, which indicates whether a district becomes a hotspot in the next 4-hour block. Predictors include the district identifier, temporal indicators, current crime conditions, lagged crime counts, and rolling historical averages.

## Training Process

The notebook uses `train_panel_featured.csv` and `test_panel_featured.csv` generated in Phase 2. To avoid temporal leakage, a time-based validation split is used during tuning: the earliest 80% of training dates are used for model fitting and the latest 20% of training dates are used for validation. After the best hyperparameters are selected, each model is retrained on the full training set and then evaluated on the 2025 test set.

The final exported artifacts are:

- `models/*.joblib`: trained model files
- `predictions/*_test_predictions.csv`: predicted probabilities and labels
- `reports/model_comparison_metrics.csv`: final model comparison table
- `reports/hyperparameter_tuning_results.csv`: full tuning log
- `reports/best_params.json`: selected best parameter combinations

## Hyperparameter Tuning

A compact grid search is used for each model. The selection metric is validation F1-score, because hotspot prediction is an imbalanced binary classification task and F1-score provides a more balanced view than accuracy alone. Logistic Regression tunes regularization strength and class weighting. Random Forest tunes tree depth, leaf size, and class weighting. Gradient Boosting tunes the number of estimators, learning rate, and tree depth. HistGradientBoosting tunes the number of boosting iterations, learning rate, maximum depth, and minimum leaf size.

## AI Assistance Declaration

Generative AI was used in a limited supporting role in the preparation of this notebook.

Its use was limited to helping structure the workflow, refine methodological explanations, suggest code templates for preprocessing and feature engineering, and assist with interpretation of intermediate results.

All final decisions, execution of the notebook, validation of outputs, and integration into the project were completed by the student.

In [1]:
from itertools import product
from pathlib import Path
import json
import time

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()
TRAIN_PATH = PROJECT_ROOT / 'train_panel_featured.csv'
TEST_PATH = PROJECT_ROOT / 'test_panel_featured.csv'
MODELS_DIR = PROJECT_ROOT / 'models'
PREDICTIONS_DIR = PROJECT_ROOT / 'predictions'
REPORTS_DIR = PROJECT_ROOT / 'reports'
TARGET_COL = 'target_hotspot_next_block'
DROP_COLS = {TARGET_COL, 'dataset', 'event_date'}

for path in (MODELS_DIR, PREDICTIONS_DIR, REPORTS_DIR):
    path.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(TRAIN_PATH, parse_dates=['event_date'])
test_df = pd.read_csv(TEST_PATH, parse_dates=['event_date'])

train_df.shape, test_df.shape


((504091, 17), (50347, 17))

In [2]:
feature_cols = [col for col in train_df.columns if col not in DROP_COLS]
categorical_cols = ['district', 'time_block', 'day_of_week', 'month', 'quarter', 'is_weekend']

combined = pd.concat(
    [
        train_df[feature_cols].assign(_source='train'),
        test_df[feature_cols].assign(_source='test'),
    ],
    ignore_index=True,
)

encoded = pd.get_dummies(
    combined,
    columns=[col for col in categorical_cols if col in combined.columns],
    drop_first=False,
    dtype=float,
)

train_x = encoded.loc[encoded['_source'] == 'train'].drop(columns='_source').reset_index(drop=True)
test_x = encoded.loc[encoded['_source'] == 'test'].drop(columns='_source').reset_index(drop=True)
train_y = train_df[TARGET_COL].astype(int)
test_y = test_df[TARGET_COL].astype(int)

unique_dates = np.array(sorted(train_df['event_date'].dt.normalize().unique()))
split_idx = max(1, int(len(unique_dates) * 0.8))
cutoff_date = pd.Timestamp(unique_dates[split_idx])
train_mask = (train_df['event_date'] < cutoff_date).to_numpy()
valid_mask = ~train_mask

print('Validation cutoff date:', cutoff_date.date())
print('Training rows for tuning:', int(train_mask.sum()))
print('Validation rows for tuning:', int(valid_mask.sum()))
print('Feature count after encoding:', train_x.shape[1])


Validation cutoff date: 2023-01-01
Training rows for tuning: 403236
Validation rows for tuning: 100855
Feature count after encoding: 62


In [3]:
def iter_param_grid(grid):
    keys = list(grid.keys())
    for values in product(*(grid[key] for key in keys)):
        yield dict(zip(keys, values))


def compute_metrics(y_true, probas, threshold=0.5):
    preds = (probas >= threshold).astype(int)
    return {
        'accuracy': accuracy_score(y_true, preds),
        'precision': precision_score(y_true, preds, zero_division=0),
        'recall': recall_score(y_true, preds, zero_division=0),
        'f1': f1_score(y_true, preds, zero_division=0),
        'roc_auc': roc_auc_score(y_true, probas),
    }

###v2: add new function###
def find_best_threshold(y_true, probas, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.1, 0.91, 0.05)

    best_threshold = 0.5
    best_metrics = None
    best_f1 = -1

    for thr in thresholds:
        metrics = compute_metrics(y_true, probas, threshold=thr)
        if metrics['f1'] > best_f1:
            best_f1 = metrics['f1']
            best_threshold = thr
            best_metrics = metrics

    return best_threshold, best_metrics


#

###model_specs v2###
model_specs = {
    'logistic_regression': {
        'builder': lambda params: LogisticRegression(
            max_iter=2000,
            solver='lbfgs',
            random_state=42,
            **params
        ),
        'grid': {
            'C': [0.01, 0.1, 1.0, 5.0, 10.0],
            'class_weight': [None, 'balanced'],
        },
        'scale': True,
    },

    'random_forest': {
        'builder': lambda params: RandomForestClassifier(
            random_state=42,
            n_jobs=-1,
            **params
        ),
        'grid': {
            'n_estimators': [100, 200, 400],
            'max_depth': [8, 12, 16, None],
            'min_samples_leaf': [1, 2, 5, 10],
            'min_samples_split': [2, 5, 10],
            'max_features': ['sqrt', 'log2', 0.5],
            'class_weight': [None, 'balanced', 'balanced_subsample'],
        },
        'scale': False,
    },

    'gradient_boosting': {
        'builder': lambda params: GradientBoostingClassifier(
            random_state=42,
            **params
        ),
        'grid': {
            'n_estimators': [100, 200],
            'learning_rate': [0.03, 0.05, 0.1],
            'max_depth': [2, 3],
            'subsample': [0.8, 1.0],
            'min_samples_leaf': [1, 5, 10],
        },
        'scale': False,
    },

    'hist_gradient_boosting': {
        'builder': lambda params: HistGradientBoostingClassifier(
            random_state=42,
            early_stopping=False,
            **params
        ),
        'grid': {
            'max_iter': [100, 200, 300],
            'learning_rate': [0.03, 0.05, 0.1],
            'max_depth': [4, 6, 8, None],
            'min_samples_leaf': [10, 20, 50],
            'l2_regularization': [0.0, 0.1, 1.0],
        },
        'scale': False,
    },
}


def fit_and_export_all_models(train_x, test_x, train_y, test_y):
    tuning_rows = []
    metric_rows = []
    best_params = {}
    feature_names = list(train_x.columns)

    for model_name, spec in model_specs.items():
        print(f'Running {model_name}...')

        scaler = None
        if spec['scale']:
            scaler = StandardScaler()
            scaler.fit(train_x.loc[train_mask])
            x_train_part = scaler.transform(train_x.loc[train_mask]).astype(np.float32)
            x_valid = scaler.transform(train_x.loc[valid_mask]).astype(np.float32)
        else:
            x_train_part = train_x.loc[train_mask].to_numpy(dtype=np.float32)
            x_valid = train_x.loc[valid_mask].to_numpy(dtype=np.float32)

        y_train_part = train_y.loc[train_mask]
        y_valid = train_y.loc[valid_mask]

        best_score = -1.0
        best_row = None

        for params in iter_param_grid(spec['grid']):
            started = time.time()
            model = spec['builder'](params)
            model.fit(x_train_part, y_train_part)
            # valid_proba = model.predict_proba(x_valid)[:, 1]
            # metrics = compute_metrics(y_valid, valid_proba)
            # row = {
            #     'model': model_name,
            #     'fit_seconds': round(time.time() - started, 2),
            #     **params,
            #     **{f'valid_{k}': v for k, v in metrics.items()},
            # }
            # tuning_rows.append(row)
            # if metrics['f1'] > best_score:
            #     best_score = metrics['f1']
            #     best_row = row

            ###training loop v2###
            ###F1默认0.5-->扫一遍threshold，最佳下的F1选Best params
            valid_proba = model.predict_proba(x_valid)[:, 1]
            best_threshold, best_valid_metrics = find_best_threshold(y_valid, valid_proba)

            row = {
                'model': model_name,
                'fit_seconds': round(time.time() - started, 2),
                'best_threshold': best_threshold,
                **params,
                **{f'valid_{k}': v for k, v in best_valid_metrics.items()},
            }
            tuning_rows.append(row)
            if best_valid_metrics['f1'] > best_score:
                best_score = best_valid_metrics['f1']
                best_row = row



        # best_params[model_name] = {key: best_row[key] for key in spec['grid'].keys()}

        ###best params v2: 还保存best threshold
        best_params[model_name] = {
            'params': {key: best_row[key] for key in spec['grid'].keys()},
            'threshold': best_row['best_threshold'],
        }

        final_scaler = None
        if spec['scale']:
            final_scaler = StandardScaler()
            final_scaler.fit(train_x)
            x_train_full = final_scaler.transform(train_x).astype(np.float32)
            x_test = final_scaler.transform(test_x).astype(np.float32)
        else:
            x_train_full = train_x.to_numpy(dtype=np.float32)
            x_test = test_x.to_numpy(dtype=np.float32)

        # final_model = spec['builder'](best_params[model_name])

        ###v2
        final_model = spec['builder'](best_params[model_name]['params'])

        started = time.time()
        final_model.fit(x_train_full, train_y)
        test_proba = final_model.predict_proba(x_test)[:, 1]
        # test_metrics = compute_metrics(test_y, test_proba)

        ###v2
        best_threshold = best_params[model_name]['threshold']
        test_metrics = compute_metrics(test_y, test_proba, threshold=best_threshold)

        metric_rows.append({
            'model': model_name,
            'cutoff_date': str(cutoff_date.date()),
            'train_rows': int(train_mask.sum()),
            'valid_rows': int(valid_mask.sum()),
            'test_rows': int(len(test_y)),
            'fit_seconds': round(time.time() - started, 2),
            **best_params[model_name],
            **{f'test_{k}': v for k, v in test_metrics.items()},
        })

        artifact = {
            'model': final_model,
            'scaler': final_scaler,
            'feature_names': feature_names,
            'best_params': best_params[model_name],
        }
        joblib.dump(artifact, MODELS_DIR / f'{model_name}.joblib')

        prediction_df = test_df[['district', 'event_date', 'time_block', TARGET_COL]].copy()
        prediction_df['predicted_probability'] = test_proba
        # prediction_df['predicted_label'] = (test_proba >= 0.5).astype(int)

        ###v2
        prediction_df['predicted_label'] = (test_proba >= best_threshold).astype(int)
        prediction_df['decision_threshold'] = best_threshold

        prediction_df.rename(columns={TARGET_COL: 'actual_label'}, inplace=True)
        prediction_df.to_csv(PREDICTIONS_DIR / f'{model_name}_test_predictions.csv', index=False)

    tuning_df = pd.DataFrame(tuning_rows).sort_values(['model', 'valid_f1', 'valid_roc_auc'], ascending=[True, False, False])
    metrics_df = pd.DataFrame(metric_rows).sort_values(['test_f1', 'test_roc_auc'], ascending=[False, False])

    tuning_df.to_csv(REPORTS_DIR / 'hyperparameter_tuning_results.csv', index=False)
    metrics_df.to_csv(REPORTS_DIR / 'model_comparison_metrics.csv', index=False)
    (REPORTS_DIR / 'best_params.json').write_text(json.dumps(best_params, indent=2), encoding='utf-8')

    return tuning_df, metrics_df, best_params


In [6]:
tuning_results, metrics, best_params = fit_and_export_all_models(train_x, test_x, train_y, test_y)
metrics


Running logistic_regression...


KeyboardInterrupt: 

## 3. Post-Training Output Saving and Organization

### Why this section is needed
The core training and tuning step has already been completed in the previous cell. At this stage, the goal is not to retrain the models, but to organize the generated outputs into a clear handoff package for later submission, reporting, and model comparison.

### What this section does
- confirms that the trained model files have been saved
- confirms that the prediction result files have been saved
- reloads the exported summary files when needed
- presents a compact comparison table for reporting

### Output
- trained model artifacts in `models/`
- prediction files in `predictions/`
- metrics and tuning summaries in `reports/`


In [ ]:
model_files = sorted(MODELS_DIR.glob('*.joblib'))
prediction_files = sorted(PREDICTIONS_DIR.glob('*_test_predictions.csv'))
report_files = sorted(REPORTS_DIR.glob('*'))

print('Saved model files:')
for path in model_files:
    print('-', path.name)

print('\nSaved prediction files:')
for path in prediction_files:
    print('-', path.name)

print('\nSaved report files:')
for path in report_files:
    print('-', path.name)


Saved model files:
- gradient_boosting.joblib
- hist_gradient_boosting.joblib
- logistic_regression.joblib
- random_forest.joblib

Saved prediction files:
- gradient_boosting_test_predictions.csv
- hist_gradient_boosting_test_predictions.csv
- logistic_regression_test_predictions.csv
- random_forest_test_predictions.csv

Saved report files:
- best_params.json
- hyperparameter_tuning_results.csv
- model_comparison_metrics.csv


### 3.1 Review the Final Model Comparison Table

We now review the final comparison table exported after tuning and full training. This table is the main reference for discussing which model performed best on the 2025 test set.


In [ ]:
metrics = pd.read_csv(REPORTS_DIR / 'model_comparison_metrics.csv')
metrics


,model,cutoff_date,train_rows,valid_rows,test_rows,fit_seconds,C,class_weight,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,n_estimators,max_depth,min_samples_leaf,learning_rate,subsample,max_iter
0,random_forest,2023-01-01,403236,100855,50347,92.70,NaN,balanced_subsample,0.743282,0.392064,0.764816,0.518389,0.827048,100.0,12.0,1.0,NaN,NaN,NaN
1,logistic_regression,2023-01-01,403236,100855,50347,2.66,1.0,balanced,0.731444,0.379558,0.766795,0.507772,0.821729,NaN,NaN,NaN,NaN,NaN,NaN
2,hist_gradient_boosting,2023-01-01,403236,100855,50347,7.66,NaN,NaN,0.834091,0.570856,0.328642,0.417138,0.833516,NaN,8.0,50.0,0.1,NaN,120.0
3,gradient_boosting,2023-01-01,403236,100855,50347,107.26,NaN,NaN,0.831172,0.566689,0.277955,0.372971,0.821238,100.0,2.0,NaN,0.1,0.8,NaN


### 3.2 Review the Hyperparameter Tuning Log

The tuning log keeps every tested hyperparameter combination together with its validation performance. This makes the tuning process transparent and gives a concrete basis for the report discussion.


In [ ]:
tuning_results = pd.read_csv(REPORTS_DIR / 'hyperparameter_tuning_results.csv')
tuning_results.head(12)


,model,fit_seconds,C,class_weight,valid_accuracy,valid_precision,valid_recall,valid_f1,valid_roc_auc,n_estimators,max_depth,min_samples_leaf,learning_rate,subsample,max_iter
0,gradient_boosting,80.48,NaN,NaN,0.795122,0.583137,0.373789,0.455563,0.814000,100.0,2.0,NaN,0.10,0.8,NaN
1,gradient_boosting,80.42,NaN,NaN,0.794804,0.597892,0.321256,0.417944,0.811968,100.0,2.0,NaN,0.05,0.8,NaN
2,hist_gradient_boosting,6.44,NaN,NaN,0.802538,0.600753,0.414173,0.490313,0.825704,NaN,8.0,50.0,0.10,NaN,120.0
3,hist_gradient_boosting,6.60,NaN,NaN,0.802469,0.600666,0.413568,0.489860,0.825741,NaN,8.0,20.0,0.10,NaN,120.0
4,hist_gradient_boosting,7.94,NaN,NaN,0.801735,0.600320,0.405180,0.483814,0.824543,NaN,8.0,50.0,0.05,NaN,120.0
5,hist_gradient_boosting,15.42,NaN,NaN,0.801586,0.599974,0.404402,0.483147,0.824636,NaN,8.0,20.0,0.05,NaN,120.0
6,logistic_regression,2.24,1.0,balanced,0.711219,0.429773,0.793411,0.557539,0.815695,NaN,NaN,NaN,NaN,NaN,NaN
7,logistic_regression,2.14,0.1,balanced,0.711229,0.429776,0.793324,0.557521,0.815698,NaN,NaN,NaN,NaN,NaN,NaN
8,logistic_regression,2.02,0.1,NaN,0.795598,0.590689,0.353857,0.442582,0.815326,NaN,NaN,NaN,NaN,NaN,NaN
9,logistic_regression,1.90,1.0,NaN,0.795548,0.590476,0.353857,0.442522,0.815331,NaN,NaN,NaN,NaN,NaN,NaN


### 3.3 Inspect the Selected Best Parameters

The selected best hyperparameters for each model are stored separately so that the final training configuration is explicit and easy to reproduce later.


In [ ]:
best_params = json.loads((REPORTS_DIR / 'best_params.json').read_text(encoding='utf-8'))
best_params


{'logistic_regression': {'C': 1.0, 'class_weight': 'balanced'},
 'random_forest': {'n_estimators': 100,
  'max_depth': 12,
  'min_samples_leaf': 1,
  'class_weight': 'balanced_subsample'},
 'gradient_boosting': {'n_estimators': 100,
  'learning_rate': 0.1,
  'max_depth': 2,
  'subsample': 0.8},
 'hist_gradient_boosting': {'max_iter': 120,
  'learning_rate': 0.1,
  'max_depth': 8,
  'min_samples_leaf': 50}}

### 3.4 Inspect a Sample Prediction File

Each prediction file contains the district, date, time block, true label, predicted probability, and final predicted class. This format is suitable for both evaluation and downstream error analysis.


In [ ]:
sample_prediction = pd.read_csv(PREDICTIONS_DIR / 'logistic_regression_test_predictions.csv')
sample_prediction.head()


,district,event_date,time_block,actual_label,predicted_probability,predicted_label
0,1,2025-01-01,0,0,0.057924,0
1,1,2025-01-01,1,0,0.416616,0
2,1,2025-01-01,2,1,0.544356,1
3,1,2025-01-01,3,0,0.775660,1
4,1,2025-01-01,4,0,0.542963,1


## 4. Report Writing Support

### Why this section is needed
To keep the final report consistent with the notebook outputs, this section generates ready-to-use report text for the required parts: `Model Implementation`, `Training Process`, and `Hyperparameter Tuning`. The text is based on the actual workflow and saved outputs from this notebook.

### Output
- a structured markdown file containing report-ready paragraphs
- a notebook preview of the generated report text


In [ ]:
best_model_row = metrics.sort_values(['test_f1', 'test_roc_auc'], ascending=[False, False]).iloc[0]
best_model_name = best_model_row['model']
feature_count = train_x.shape[1]

report_text = f"""# Model Implementation

Four classification models were implemented for hotspot prediction: Logistic Regression, Random Forest, Gradient Boosting, and HistGradientBoosting. These models were selected to provide a balanced comparison between an interpretable linear baseline and more flexible ensemble-based classifiers. Logistic Regression was included as a simple and transparent reference model, while the tree-based ensemble models were included because they can capture non-linear relationships among spatial, temporal, and historical crime features. The modeling target was `target_hotspot_next_block`, which indicates whether a district becomes a hotspot in the next 4-hour block.

The model input was based on the district-level featured panel prepared in Phase 2. After excluding the target column, dataset marker, and raw date field from direct model input, the feature matrix contained {feature_count} encoded predictors. These predictors included district identity, time block, day of week, month, quarter, weekend indicator, current crime count, current hotspot status, lagged counts from previous blocks, and rolling historical averages. Categorical fields were converted using one-hot encoding so that all four models could be trained on a consistent feature representation.

# Training Process

The training workflow used `train_panel_featured.csv` as the development dataset and `test_panel_featured.csv` as the final evaluation dataset. To preserve the chronological structure of the crime forecasting task, a temporal validation strategy was used instead of a random train-validation split. The earliest 80% of training dates were used for fitting candidate models during tuning, while the most recent 20% of training dates were reserved for validation. This design reduces the risk of information leakage from future periods into earlier observations and better matches the intended real-world prediction setting.

After selecting the best hyperparameters for each model, the chosen configuration was retrained on the full 2015-2024 training dataset and then applied to the 2025 test set. For each model, the notebook saved a serialized model artifact in the `models/` folder and a corresponding prediction file in the `predictions/` folder. Final model performance was compared using accuracy, precision, recall, F1-score, and ROC-AUC. Among the evaluated models, `{best_model_name}` achieved the strongest overall test performance under the current selection rule based on F1-score.

# Hyperparameter Tuning

Hyperparameter tuning was carried out through a compact grid search for each of the four models. For Logistic Regression, the search focused on regularization strength and class weighting. For Random Forest, the tuning considered tree depth and minimum leaf size under a balanced-subsample setting. For Gradient Boosting, the search varied the learning rate while keeping a stable number of boosting stages and shallow trees. For HistGradientBoosting, the tuning considered the learning rate and minimum leaf size under a fixed iteration budget and tree depth.

Each hyperparameter combination was evaluated on the temporal validation subset, and the parameter set with the highest validation F1-score was selected as the final configuration for that model. All tested combinations were exported to `reports/hyperparameter_tuning_results.csv`, and the selected parameter sets were exported to `reports/best_params.json`. This makes the tuning process transparent, reproducible, and directly traceable to the saved notebook outputs."""

report_path = REPORTS_DIR / 'model_training_report_sections.md'
report_path.write_text(report_text, encoding='utf-8')
print(f'Saved report section draft to: {report_path}')
print()
print(report_text)


Saved report section draft to: d:\Documents\NUS\sem2\IT5006\project\reports\model_training_report_sections.md

# Model Implementation

Four classification models were implemented for hotspot prediction: Logistic Regression, Random Forest, Gradient Boosting, and HistGradientBoosting. These models were selected to provide a balanced comparison between an interpretable linear baseline and more flexible ensemble-based classifiers. Logistic Regression was included as a simple and transparent reference model, while the tree-based ensemble models were included because they can capture non-linear relationships among spatial, temporal, and historical crime features. The modeling target was `target_hotspot_next_block`, which indicates whether a district becomes a hotspot in the next 4-hour block.

The model input was based on the district-level featured panel prepared in Phase 2. After excluding the target column, dataset marker, and raw date field from direct model input, the feature matrix con